In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("recipes.csv")
df.head(3)

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen ...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_...",...,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and..."
2,40,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05T19:52:00Z,This is from one of my first Good House Keepi...,"c(""https://img.sndimg.com/food/image/upload/w_...",...,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0,NaN,"c(""Into a 1 quart Jar with tight fitting lid, ..."


In [7]:
print(df.columns.tolist())

['RecipeId', 'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientQuantities', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions']


In [8]:
# find fried rice recipe
fried_rice_df = df[df['Name'].astype(str).str.contains('fried rice', case=False, na=False)].copy()

# review bot reviews or dead reviews
fried_rice_df = fried_rice_df.dropna(subset=['AggregatedRating', 'ReviewCount'])
fried_rice_df = fried_rice_df[fried_rice_df['ReviewCount'] > 0]

print(f"Total fried rice recipes with reviews: {len(fried_rice_df)}")

Total fried rice recipes with reviews: 418


In [11]:
C = fried_rice_df['AggregatedRating'].mean()
m = fried_rice_df['ReviewCount'].quantile(0.50)

def weighted_rating(row, m=m, C=C):
    v = row['ReviewCount']
    R = row['AggregatedRating']
    return (v / (v + m)) * R + (m / (v + m)) * C

fried_rice_df['weighted_score'] = fried_rice_df.apply(weighted_rating, axis=1)

# top 5 ranked recipes
fried_rice_df[['Name', 'AggregatedRating', 'ReviewCount', 'weighted_score']].sort_values(
    by='weighted_score', ascending=False
).head(5)

,Name,AggregatedRating,ReviewCount,weighted_score
35045,Chinese Fried Rice,5.0,539.0,4.998603
5623,Fried Rice,5.0,248.0,4.996976
17294,Easy Fried Rice,5.0,93.0,4.992042
335007,Kittencal's Best Chinese Fried Rice With Egg,5.0,92.0,4.991958
66898,Benihana Japanese Fried Rice,5.0,88.0,4.991600


In [12]:
fried_rice_df['clean_ingredients'] = fried_rice_df['RecipeIngredientParts'].astype(str).str.lower()

components = {
    'has_sesame_oil': r'sesame oil',
    'has_garlic': r'garlic',
    'has_ginger': r'ginger',
    'has_green_onion': r'scallion|green onion|spring onion',
    'has_soy_sauce': r'soy sauce',
    'has_dark_soy': r'dark soy',
    'has_oyster_sauce': r'oyster sauce',
    'has_msg': r'msg|monosodium glutamate',
    'has_butter': r'butter',
    'has_egg': r'egg'
}

for col, pattern in components.items():
    fried_rice_df[col] = fried_rice_df['clean_ingredients'].str.contains(pattern, regex=True).astype(int)

# how frequently each ingredient appears
fried_rice_df[[col for col in components.keys()]].mean().sort_values(ascending=False)

has_soy_sauce       0.767943
has_egg             0.710526
has_green_onion     0.569378
has_garlic          0.535885
has_ginger          0.169856
has_butter          0.095694
has_oyster_sauce    0.071770
has_dark_soy        0.019139
has_msg             0.011962
has_sesame_oil      0.000000
dtype: float64

In [18]:
components = {
    # protien
    'has_chicken': r'chicken',
    'has_pork': r'pork|bacon|ham',
    'has_chinese_sausage': r'sausage|lap cheong|lop chong|la-ap-chieng',
    'has_shrimp': r'shrimp|prawn',
    'has_beef': r'beef|steak',
    'has_egg': r'egg',
    'has_tofu': r'tofu',
    
    # seasoning
    'has_garlic': r'garlic',
    'has_ginger': r'ginger',
    'has_green_onion': r'scallion|green onion|spring onion',
    'has_soy_sauce': r'soy sauce',
    'has_dark_soy': r'dark soy',
    'has_sesame_oil': r'sesame oil',
    'has_oyster_sauce': r'oyster sauce',
    'has_msg': r'msg|monosodium glutamate',
    'has_butter': r'butter',
    'has_white_pepper': r'white pepper',
    
    # mix-ins
    'has_peas_carrots': r'pea|carrot',
    'has_kimchi': r'kimchi'
}

# gen 1/0 indicator columns
fried_rice_df['clean_ingredients'] = fried_rice_df['RecipeIngredientParts'].astype(str).str.lower()
for col, pattern in components.items():
    fried_rice_df[col] = fried_rice_df['clean_ingredients'].str.contains(pattern, regex=True).astype(int)

In [19]:
comparison_rows = []

for feature in components.keys():
    group_with = fried_rice_df[fried_rice_df[feature] == 1]['weighted_score']
    group_without = fried_rice_df[fried_rice_df[feature] == 0]['weighted_score']
    
    comparison_rows.append({
        'ingredient': feature.replace('has_', ''),
        'count_with': len(group_with),
        'count_without': len(group_without),
        'mean_with': group_with.mean(),
        'mean_without': group_without.mean(),
        'lift': group_with.mean() - group_without.mean()
    })

comparison_df = pd.DataFrame(comparison_rows)

# sort by highest lift (positive impact on weighted score)
comparison_df.sort_values(by='lift', ascending=False).reset_index(drop=True)

,ingredient,count_with,count_without,mean_with,mean_without,lift
0,oyster_sauce,30,388,4.760200,4.648686,0.111515
1,soy_sauce,321,97,4.676732,4.590361,0.086372
2,egg,297,121,4.680249,4.598861,0.081387
3,white_pepper,17,401,4.717154,4.654126,0.063028
4,green_onion,238,180,4.682368,4.622737,0.059631
5,beef,19,399,4.679185,4.655618,0.023567
6,tofu,10,408,4.679349,4.656134,0.023215
7,garlic,224,194,4.667217,4.644533,0.022684
8,pork,99,319,4.673403,4.651502,0.021901
9,shrimp,61,357,4.673818,4.653763,0.020056


In [21]:
%pip install scipy

   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---- ----------------------------------- 4.2/37.4 MB 21.6 MB/s eta 0:00:02
   ----------- ---------------------------- 10.5/37.4 MB 25.9 MB/s eta 0:00:02
   -------------- ------------------------- 13.9/37.4 MB 21.8 MB/s eta 0:00:02
   ----------------------- ---------------- 21.8/37.4 MB 25.5 MB/s eta 0:00:01
   -------------------------------- ------- 30.1/37.4 MB 28.3 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 29.5 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 26.4 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\amwom\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [22]:
from scipy import stats

p_values = []
for feature in components.keys():
    group_with = fried_rice_df[fried_rice_df[feature] == 1]['weighted_score']
    group_without = fried_rice_df[fried_rice_df[feature] == 0]['weighted_score']
    
    # Run two-sample t-test (assumes unequal variances via Welch's t-test)
    if len(group_with) > 5 and len(group_without) > 5:
        _, p_val = stats.ttest_ind(group_with, group_without, equal_var=False, nan_policy='omit')
    else:
        p_val = np.nan
        
    p_values.append(p_val)

comparison_df['p_value'] = p_values
comparison_df['stat_significant (< 0.05)'] = comparison_df['p_value'] < 0.05

# view sorted by lift
comparison_df.sort_values(by='lift', ascending=False).reset_index(drop=True)

,ingredient,count_with,count_without,mean_with,mean_without,lift,p_value,stat_significant (< 0.05)
0,oyster_sauce,30,388,4.760200,4.648686,0.111515,0.007932,True
1,soy_sauce,321,97,4.676732,4.590361,0.086372,0.029247,True
2,egg,297,121,4.680249,4.598861,0.081387,0.017824,True
3,white_pepper,17,401,4.717154,4.654126,0.063028,0.259993,False
4,green_onion,238,180,4.682368,4.622737,0.059631,0.046305,True
5,beef,19,399,4.679185,4.655618,0.023567,0.732278,False
6,tofu,10,408,4.679349,4.656134,0.023215,0.608453,False
7,garlic,224,194,4.667217,4.644533,0.022684,0.434650,False
8,pork,99,319,4.673403,4.651502,0.021901,0.485185,False
9,shrimp,61,357,4.673818,4.653763,0.020056,0.617272,False


In [24]:
# sesame appears in any form
print(fried_rice_df['clean_ingredients'].str.contains('sesame').sum())

17


In [27]:
# OLS, checking for multicollinearity and test which ingredients remain significant while holding all others constant

import statsmodels.api as sm

X_cols = ['has_oyster_sauce', 'has_soy_sauce', 'has_egg', 'has_green_onion', 'has_garlic', 'has_ginger']
X = fried_rice_df[X_cols]
X = sm.add_constant(X)
y = fried_rice_df['weighted_score']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         weighted_score   R-squared:                       0.044
Model:                            OLS   Adj. R-squared:                  0.030
Method:                 Least Squares   F-statistic:                     3.128
Date:                Tue, 13 Oct 2026   Prob (F-statistic):            0.00522
Time:                        02:09:14   Log-Likelihood:                -70.200
No. Observations:                 418   AIC:                             154.4
Df Residuals:                     411   BIC:                             182.6
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                4.5172      0.039  

In [28]:
top_recipes = fried_rice_df.sort_values(by='weighted_score', ascending=False).head(5)
display(top_recipes[['Name', 'weighted_score', 'ReviewCount', 'AggregatedRating']])

,Name,weighted_score,ReviewCount,AggregatedRating
35045,Chinese Fried Rice,4.998603,539.0,5.0
5623,Fried Rice,4.996976,248.0,5.0
17294,Easy Fried Rice,4.992042,93.0,5.0
335007,Kittencal's Best Chinese Fried Rice With Egg,4.991958,92.0,5.0
66898,Benihana Japanese Fried Rice,4.991600,88.0,5.0


In [26]:
%pip install statsmodels

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   -------------- ------------------------- 4.2/11.3 MB 21.5 MB/s eta 0:00:01
   -------------------------------- ------- 9.2/11.3 MB 22.7 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 20.4 MB/s  0:00:00

   ---------------------------------------- 0/6 [wrapt]
   ---------------------------------------- 0/6 [wrapt]
   ---------------------------------------- 0/6 [wrapt]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------ --------------------------------- 1/6 [patsy]
   ------------- --------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\amwom\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [29]:
# Filter to top 25% of recipes
top_quartile = fried_rice_df[fried_rice_df['weighted_score'] >= fried_rice_df['weighted_score'].quantile(0.75)]

# Inspect co-occurrence of statistically significant ingredients
combo_cols = ['has_oyster_sauce', 'has_soy_sauce', 'has_egg', 'has_green_onion', 'has_garlic']
display(top_quartile[combo_cols].mean().sort_values(ascending=False).to_frame(name='prevalence_in_top_25%'))

# Check how many of the top 25% use the 'Holy Trinity' (Soy Sauce + Egg + Green Onion + Oyster Sauce)
all_core = (top_quartile['has_oyster_sauce'] == 1) & \
           (top_quartile['has_soy_sauce'] == 1) & \
           (top_quartile['has_egg'] == 1) & \
           (top_quartile['has_green_onion'] == 1)

print(f"Percentage of top-tier recipes using all 4 significant ingredients: {all_core.mean() * 100:.1f}%")

,prevalence_in_top_25%
has_soy_sauce,0.828829
has_egg,0.801802
has_green_onion,0.621622
has_garlic,0.558559
has_oyster_sauce,0.108108


Percentage of top-tier recipes using all 4 significant ingredients: 2.7%


In [30]:
# find recipes that include oyster sauce, soy sauce, egg, and green onion
golden_recipes = fried_rice_df[
    (fried_rice_df['has_oyster_sauce'] == 1) & 
    (fried_rice_df['has_soy_sauce'] == 1) & 
    (fried_rice_df['has_egg'] == 1)
].sort_values(by='weighted_score', ascending=False)

# inspect the top 3 matches' exact ingredient parts and quantities
for idx, row in golden_recipes.head(3).iterrows():
    print(f"--- {row['Name']} (Score: {row['weighted_score']:.2f}) ---")
    print("Parts:", row['RecipeIngredientParts'])
    print("Quantities:", row['RecipeIngredientQuantities'])
    print()

--- Thai Spicy Basil Fried Rice (Score: 4.96) ---
Parts: c("oyster sauce", "fish sauce", "soy sauce", "white sugar", "olive oil flavored cooking spray", "cooked jasmine rice", "garlic cloves", "red bell pepper", "onion", "egg substitute", "shrimp", "Thai basil", "cilantro")
Quantities: c("3", "2", "1", "1", NA, "4", "6", "2", "1", "1", "1/2", "8", "2", "1/2")

--- Mrs. Seiko's Fried Rice (Score: 4.95) ---
Parts: c("canola oil", "green onions", "shrimp", "rice", "light soy sauce", "oyster sauce", "eggs")
Quantities: c("2 -3", "3", "2", "4", "2 -3", "1  - 1 1/2", "2 -3", NA)

--- Pork (Chicken, or Shrimp) Fried Rice (Score: 4.95) ---
Parts: c("instant brown rice", "water", "beef bouillon cube", "chicken bouillon cube", "soy sauce", "oyster sauce", "canola oil", "chicken", "shrimp", "green onions", "eggs")
Quantities: c("2", "1 3/4", "1", "1", "2", "2", "1/2", "2", "1", "1/2", "1/2", "4", "3")



In [31]:
import re
import numpy as np
import pandas as pd

def parse_r_vector(text):
    if pd.isna(text):
        return []
    # extract quoted elements or comma-separated tokens inside c(...)
    items = re.findall(r'"([^"]*)"', str(text))
    if not items:
        # Fallback for unquoted vectors like c(1, 2, NA)
        inner = re.sub(r'^c\(|\)$', '', str(text))
        items = [x.strip() for x in inner.split(',')]
    return items

def parse_quantity(val):
    if not val or val in ['NA', 'None', '']:
        return np.nan
    # handle ranges like '2 -3' or '1 - 1 1/2' by taking the average
    if '-' in val:
        parts = [p.strip() for p in val.split('-') if p.strip()]
        vals = [parse_quantity(p) for p in parts]
        vals = [v for v in vals if not np.isnan(v)]
        return np.mean(vals) if vals else np.nan
    # handle mixed fractions like '1 1/2'
    parts = val.strip().split()
    total = 0.0
    for p in parts:
        if '/' in p:
            num, denom = p.split('/')
            total += float(num) / float(denom)
        else:
            try:
                total += float(p)
            except ValueError:
                return np.nan
    return total

records = []
for idx, row in golden_recipes.iterrows():
    parts = parse_r_vector(row['RecipeIngredientParts'])
    quants = parse_r_vector(row['RecipeIngredientQuantities'])
    
    if len(parts) == len(quants):
        mapping = dict(zip([p.lower() for p in parts], [parse_quantity(q) for q in quants]))
        
        soy_val = next((v for k, v in mapping.items() if 'soy sauce' in k), np.nan)
        oyster_val = next((v for k, v in mapping.items() if 'oyster sauce' in k), np.nan)
        rice_val = next((v for k, v in mapping.items() if 'rice' in k), np.nan)
        
        records.append({
            'name': row['Name'],
            'score': row['weighted_score'],
            'soy_amount': soy_val,
            'oyster_amount': oyster_val,
            'rice_amount': rice_val
        })

ratio_df = pd.DataFrame(records).dropna(subset=['soy_amount', 'oyster_amount'])
ratio_df['soy_to_oyster_ratio'] = ratio_df['soy_amount'] / ratio_df['oyster_amount']

# summary
print("Median Soy Sauce amount (tbsp):", ratio_df['soy_amount'].median())
print("Median Oyster Sauce amount (tbsp):", ratio_df['oyster_amount'].median())
print("Median Soy : Oyster Ratio:", round(ratio_df['soy_to_oyster_ratio'].median(), 2))
print("Median Rice amount (cups):", ratio_df['rice_amount'].median())

Median Soy Sauce amount (tbsp): 2.0
Median Oyster Sauce amount (tbsp): 2.25
Median Soy : Oyster Ratio: 0.9
Median Rice amount (cups): 3.5


In [32]:
# 1. Export the main fried rice dataset with scores and binary flags
fried_rice_df.to_csv("best fried rice datasci/fried_rice_processed.csv", index=False)

# 2. Export your statistical lift table
comparison_df.to_csv("best fried rice datasci/ingredient_lift.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'best fried rice datasci'

In [33]:
# 1. Export main fried rice dataframe
fried_rice_df.to_csv("fried_rice_processed.csv", index=False)

# 2. Export the statistical lift table
comparison_df.to_csv("ingredient_lift.csv", index=False)